In [1]:
!pip install -U langchain-community
!pip install pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.3/302.3 kB 5.7 MB/s eta 0:00:00


Lectura PDF

In [2]:
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Cargar el PDF desde ,las carpetas de Drive
pdf_path = "./el principito.pdf"
loader = PyPDFLoader(pdf_path)
documents = loader.load()

# Extraer el texto de cada página
text = "\n".join([doc.page_content for doc in documents])

# Chunks

In [3]:
# Dividir en fragmentos (chunks)
text_splitter = RecursiveCharacterTextSplitter(chunk_size= 500, chunk_overlap = 200)

chunks = text_splitter.split_text(text)  # revisar como usar .split_docuement para que sea compatible con chromaDB

# Transformar texto y vectores

In [4]:
from sentence_transformers import SentenceTransformer

def text_to_vector(text):

    # Cargar el modelo de embeddings
    model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

    # Convertir el texto en un vector numérico
    vector = model.encode(text)

    return vector

def vector_to_text(vector):
  # Cargar modelo
  model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

  return text

frase_buscar = "Lo esencial es invisible a los ojos."

frase_vector = text_to_vector(frase_buscar)

frase_vector = frase_vector.astype(float).tolist()

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

# Chunks en vectores

In [5]:
chunk_vectors = [text_to_vector(chunk) for chunk in chunks]
print(len(chunk_vectors))

264


# Guardar Chunks

In [6]:
# import numpy as np

# # Convertir a un array de NumPy y guardar
# np.save("chunks.npy", np.array(chunk_vectors))

# # Leer desde el archivo .npy
# loaded_vectors = np.load("chunks.npy")

# print("Vectores cargados:", loaded_vectors)

# Instalar SWIG

In [7]:
!apt-get install swig

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  swig4.0
Suggested packages:
  swig-doc swig-examples swig4.0-examples swig4.0-doc
The following NEW packages will be installed:
  swig swig4.0
0 upgraded, 2 newly installed, 0 to remove and 30 not upgraded.
Need to get 1,116 kB of archives.
After this operation, 5,542 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 swig4.0 amd64 4.0.2-1ubuntu1 [1,110 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 swig all 4.0.2-1ubuntu1 [5,632 B]
Fetched 1,116 kB in 2s (721 kB/s)
Selecting previously unselected package swig4.0.
(Reading database ... 126210 files and directories currently installed.)
Preparing to unpack .../swig4.0_4.0.2-1ubuntu1_amd64.deb ...
Unpacking swig4.0 (4.0.2-1ubuntu1) ...
Selecting previously unselected package swig.
Preparing to unpack .../swig_4.0.2-1ubunt

# Ball Tree

## Headers

### Se definen qué métodos son públicos y privados

In [8]:
%%file ball_tree.h

#ifndef BALL_TREE_H
#define BALL_TREE_H

#include <vector>

class BallTree {
private:
    struct Compare {
        int depth;
        Compare(int d) : depth(d) {}

        bool operator()(const std::vector<double>& a, const std::vector<double>& b) const {
            return a[depth % a.size()] < b[depth % a.size()];
        }
    };

    struct Node {
        std::vector<double> point;
        Node* left;
        Node* right;
    };

    Node* root;
    std::vector<std::vector<double> > data;

    // Función auxiliar para construir el árbol
    Node* build_tree(int left, int right, int depth);

    // Función auxiliar para encontrar el vecino más cercano
    void nearest_neighbor(Node* node, const std::vector<double>& target, Node*& best, double& best_dist, int depth);

public:
    BallTree(); // Constructor

    // Construye el árbol a partir de un conjunto de puntos
    void build(const std::vector<std::vector<double> >& points);

    // Encuentra el punto más cercano en el árbol
    std::vector<double> find_nearest(const std::vector<double>& target);
};

#endif // BALL_TREE_H


Writing ball_tree.h


## Class

### Clase encargada de la lógica

In [9]:
%%file ball_tree.cpp
#include "ball_tree.h"
#include <cmath>
#include <limits>
#include <algorithm>

// Función para calcular la distancia euclidiana
double euclidean_distance(const std::vector<double>& a, const std::vector<double>& b) {
    double sum = 0.0;
    for (size_t i = 0; i < a.size(); ++i) {
        sum += (a[i] - b[i]) * (a[i] - b[i]);
    }
    return std::sqrt(sum);
}

// Constructor de la clase BallTree
BallTree::BallTree() : root(nullptr) {}

// Método para construir el árbol
void BallTree::build(const std::vector<std::vector<double> >& points) {
    this->data = points;
    this->root = build_tree(0, data.size() - 1, 0);
}

// Método privado para construir el árbol
BallTree::Node* BallTree::build_tree(int left, int right, int depth) {
    if (left > right) return nullptr;

    int mid = (left + right) / 2;
    std::nth_element(data.begin() + left, data.begin() + mid, data.begin() + right + 1, BallTree::Compare(depth));

    Node* node = new Node();
    node->point = data[mid];
    node->left = build_tree(left, mid - 1, depth + 1);
    node->right = build_tree(mid + 1, right, depth + 1);

    return node;
}

// Método privado para encontrar el vecino más cercano
void BallTree::nearest_neighbor(Node* node, const std::vector<double>& target, Node*& best, double& best_dist, int depth) {
    if (!node) return;

    double dist = euclidean_distance(target, node->point);
    if (dist < best_dist) {
        best_dist = dist;
        best = node;
    }

    int axis = depth % target.size();
    Node* next = target[axis] < node->point[axis] ? node->left : node->right;
    Node* other = (next == node->left) ? node->right : node->left;

    nearest_neighbor(next, target, best, best_dist, depth + 1);

    if (std::abs(target[axis] - node->point[axis]) < best_dist) {
        nearest_neighbor(other, target, best, best_dist, depth + 1);
    }
}

// Método público para encontrar el punto más cercano
std::vector<double> BallTree::find_nearest(const std::vector<double>& target) {
    Node* best = nullptr;
    double best_dist = std::numeric_limits<double>::max();
    nearest_neighbor(root, target, best, best_dist, 0);
    return best ? best->point : std::vector<double>();
}


Writing ball_tree.cpp


## Interfaz

In [10]:
%%file ball_tree.i
%module ball_tree
%{
#include "ball_tree.h"
%}

%include "std_vector.i"
%template(VectorDouble) std::vector<double>;
%template(VectorVectorDouble) std::vector<std::vector<double>>;

%include "ball_tree.h"

// Exponer la clase BallTree a Python
%feature("director") BallTree;


Writing ball_tree.i


# Ejecutar SWIG

In [11]:
!swig -c++ -python ball_tree.i

In [12]:
!g++ -O2 -fPIC -c ball_tree.cpp

In [13]:
!g++ -O2 -fPIC -c ball_tree_wrap.cxx -I/usr/include/python3.10

In [14]:
!g++ -shared ball_tree.o ball_tree_wrap.o -o _ball_tree.so

# Usar DLL

In [28]:
import numpy as np
import ball_tree
import timeit

tree = ball_tree.BallTree()

vector_float = np.array(chunk_vectors, dtype=np.float32)

vector_float = vector_float.tolist()

tree.build(vector_float)


frase_buscar = input("Ingresa una frase: ")

frase_vector = text_to_vector(frase_buscar)
frase_vector = frase_vector.astype(float).tolist()

time_start = timeit.default_timer()

chunk_similar = tree.find_nearest(frase_vector)


# for i, chunk in enumerate(chunks):
#   if chunk_similar in chunk_vectors[i]:
#     print(chunks[i])


time_end = timeit.default_timer()
print("Tiempo de ejecución:", time_end - time_start)

Ingresa una frase: principito. Pero	el	vanidoso	no	le	oyó.	Los	vanidosos	sólo	oyen	las	alabanzas. —Tú	me	admiras	mucho,	¿verdad?	—preguntó	el	vanidoso	al	principito. —¿Qué	significa	admirar? —Admirar	significa	reconocer	que	yo	soy	el	hombre	más	bello,	el	mejor vestido,	el	más	rico	y	el	más	inteligente	del	planeta. —¡Si	tú	estás	solo	en	tu	planeta! —¡Hazme	ese	favor,	admírame	de	todas	maneras! —¡Bueno!	Te	admiro	—dijo	el	principito	encogiéndose	de	hombros—, pero	¿para	qué	te	sirve? Y	el	principito	se	marchó.
Tiempo de ejecución: 0.00015432999998665764


# Chroma

In [ ]:
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import Chroma
from langchain.embeddings import HuggingFaceEmbeddings
from sentence_transformers import SentenceTransformer
import timeit

# 📌 Cargar el PDF
pdf_path = "./el principito.pdf"
loader = PyPDFLoader(pdf_path)
documents = loader.load()  # Lista de objetos `Document`

# 🔹 Dividir en fragmentos (chunks) con metadatos
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)  # Se usa `split_documents()`, no `split_text()`

# 📌 Cargar el modelo de embeddings
embedding_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

def text_to_vector(text):
    """Convierte un texto en un embedding"""
    return embedding_model.encode(text).tolist()

# 🔹 Crear colección en ChromaDB
vectorstore = Chroma(
    collection_name="el_principito",
    embedding_function=HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2"),
)

# 🔹 Agregar documentos a ChromaDB
for i, chunk in enumerate(chunks):
    embedding = text_to_vector(chunk.page_content)
    vectorstore.add_texts(
        texts=[chunk.page_content],  # Texto del fragmento
        embeddings=[embedding],  # Embedding generado
        metadatas=[{"page": i}],  # Metadatos con el número de fragmento
        ids=[str(i)]  # ID único
    )

# 🔹 Crear un retriever para buscar documentos similares
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={'k': 1})

# 🔎 Ejemplo de búsqueda

query = input("Ingrese un texto a buscar")
start_time = timeit.default_timer()
results = retriever.get_relevant_documents(query)
chroma_time = timeit.default_timer() - start_time

print(chroma_time)

# Mostrar resultados
for doc in results:
    print(f"Página: {doc.metadata['page']}\nTexto: {doc.page_content}\n{'-'*50}")
